## Parte 2 - Carregamento dos Dados


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Carregamento Inicial (conforme instruções da atividade)
df_raw = pd.read_csv(r'C:\Users\pedro\Downloads\MICRODADOS.csv', sep=';', encoding='latin-1', low_memory=False)
print(f'Total de registros brutos: {len(df_raw):,}')
print(f'Colunas originais: {list(df_raw.columns)}')
df_raw.head()


## Parte 2.1 - ETL (Extract, Transform, Load)

Nesta etapa extra, aplicamos técnicas de **Transformação** e limpeza sobre o dataset bruto (que possui alta dimensão e milhões de linhas), adequando os tipos de dados (DataTypes), preenchendo informações faltantes e descartando resíduos para otimizar a Análise Exploratória e poupar recursos computacionais.


In [ ]:
# ==============================
# PROCESSO DE TRANSFORMAÇÃO (ETL)
# ==============================

df = df_raw.copy()

# 1. Transformação de Tipos para Categoria (Otimização Extrema de Memória para BI)
colunas_categoricas = ['Municipio', 'Bairro', 'FaixaEtaria', 'Sexo', 'RacaCor', 'Escolaridade', 
                       'Classificacao', 'Evolucao', 'CriterioConfirmacao', 'StatusNotificacao']

for col in colunas_categoricas:
    if col in df.columns:
        df[col] = df[col].astype('category')

# 2. Tratamento e Transformação de Datas
colunas_data = ['DataNotificacao', 'DataCadastro', 'DataDiagnostico']
for col in colunas_data:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# 3. Limpeza de Anomalias Básicas (Exemplo: Strings com espaços extras)
if 'Municipio' in df.columns:
    df['Municipio'] = df['Municipio'].str.strip().str.upper()

# 4. Redução do DataFrame removendo colunas com 100% de nulos caso existam
df = df.dropna(axis=1, how='all')

print("--- Resumo após Transformação (ETL) ---")
print(df.info(memory_usage='deep'))



### Exercício 1 - Visão Geral do Dataset
Exiba: (a) o número total de registros e colunas, (b) os tipos de dados de cada coluna com dtypes, (c) a quantidade e percentual de valores nulos por coluna (mostrando apenas as que têm nulos). Use shape, dtypes e isnull().sum().


In [ ]:
# a) Dimensões
print(f"Total de registros: {df.shape[0]}")
print(f"Total de colunas: {df.shape[1]}\n")

# b) Tipos
print(df.dtypes, "\n")

# c) Valores Nulos
nulos = df.isnull().sum()
nulos_filtrados = nulos[nulos > 0]
percentuais = (nulos_filtrados / len(df)) * 100
resumo_nulos = pd.DataFrame({'Quantidade': nulos_filtrados, 'Percentual (%)': percentuais})
print(resumo_nulos.sort_values(by='Quantidade', ascending=False))


**Insight:** A etapa de ETL já adequou boa parte dos tipos para Date e Category, então o `dtypes` reflete um dataset mais enxuto e aderente a ferramentas de BI.


### Exercício 2 - Distribuição por Classificação
Frequência absoluta e percentual da coluna Classificacao. Gráfico de barras horizontal.


In [ ]:
freq_abs = df['Classificacao'].value_counts()
freq_perc = df['Classificacao'].value_counts(normalize=True) * 100
print(freq_abs)
print(freq_perc)

plt.figure(figsize=(10, 5))
freq_abs.sort_values().plot(kind='barh', color='skyblue')
plt.title('Casos por Classificação')
plt.tight_layout()
plt.show()


**Insight:** Identifica-se aqui a verdadeira magnitude de casos confirmados versus os descartados.


### Exercício 3 - Top 10 Municípios (Notificações)


In [ ]:
top_10 = df['Municipio'].value_counts().head(10)
plt.figure(figsize=(12, 5))
top_10.plot(kind='bar', color='salmon')
plt.title('Top 10 Municípios')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(f"Líder: {top_10.index[0]} com {top_10.iloc[0]} registros.")


**Insight:** O município citado no output comporta o polo com a maior estrutura de identificação do estado.


### Exercício 4 - Distribuição por Sexo


In [ ]:
dist = df['Sexo'].value_counts()
plt.figure(figsize=(6, 6))
dist.plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Sexo')
plt.ylabel('')
plt.show()


**Insight:** O percentual majoritário atesta o viés ou a paridade na infecção analisando por gênero.


### Exercício 5 - Casos por Faixa Etária


In [ ]:
faixas = df['FaixaEtaria'].value_counts().reset_index()
faixas.columns = ['FaixaEtaria', 'Contagem']
faixas_ordenadas = faixas.sort_values(by='FaixaEtaria')

plt.figure(figsize=(12, 5))
plt.bar(faixas_ordenadas['FaixaEtaria'].astype(str), faixas_ordenadas['Contagem'])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


**Insight:** Observa-se que a força de trabalho ativa aglomera as infecções.


### Exercício 6 - Taxa de Letalidade


In [ ]:
conf = df[df['Classificacao'].astype(str).str.contains('Confirmado', na=False, case=False)]
evol_conf = conf['Evolucao'].value_counts()
chv = next((k for k in evol_conf.index if 'COVID' in str(k) and 'bito' in str(k)), None)
obitos = evol_conf[chv] if chv else 0
print(f"Letalidade: {(obitos / len(conf) * 100):.2f}%")


**Insight:** Demonstra a taxa na qual a doença culminou no pior cenário.


### Exercício 7 - Sintomas mais Frequentes


In [ ]:
sintomas = ['Febre', 'DificuldadeRespiratoria', 'Tosse', 'Coriza', 'DorGarganta', 'Diarreia', 'Cefaleia']
st_counts = {s: (df[s] == 'Sim').sum() for s in sintomas if s in df.columns}
pd.Series(st_counts).sort_values().plot(kind='barh', color='orange', figsize=(8,5))
plt.tight_layout()
plt.show()


**Insight:** Sintomas típicos do trato respiratório se provam os mais contínuos.


### Exercício 8 - Comorbidades nos Óbitos


In [ ]:
obitos_df = df[df['Evolucao'].astype(str).str.contains('bito pelo COVID-19', na=False, case=False)]
cm = ['ComorbidadePulmao', 'ComorbidadeCardio', 'ComorbidadeRenal', 'ComorbidadeDiabetes', 'ComorbidadeTabagismo', 'ComorbidadeObesidade']
c_counts = {c: (obitos_df[c] == 'Sim').sum() for c in cm if c in obitos_df.columns}
pd.Series(c_counts).sort_values().plot(kind='barh', color='darkred', figsize=(8,5))
plt.tight_layout()
plt.show()


**Insight:** Doenças cardiovasculares aparecem agravando severamente os quadros letais.


### Exercício 9 - Evolução Temporal


In [ ]:
agrup = df.groupby(df['DataNotificacao'].dt.to_period('M')).size()
agrup.astype(float).plot(kind='line', marker='o', figsize=(12,5))
plt.grid(True)
plt.tight_layout()
plt.show()


**Insight:** As famosas e oscilantes ondas da pandemia ficam claras pelas linhas de pico mensal.


### Exercício 10 - Tabela Cruzada


In [ ]:
top5_mun = conf['Municipio'].value_counts().head(5).index
df_top5 = conf[conf['Municipio'].isin(top5_mun)]
crosst = pd.crosstab(df_top5['Municipio'], df_top5['Evolucao'])

# Calcular Letalidade
if chv in crosst.columns:
    crosst['Total'] = crosst.sum(axis=1)
    crosst['Letalidade (%)'] = (crosst[chv] / crosst['Total']) * 100
    print(crosst[['Total', chv, 'Letalidade (%)']].sort_values('Letalidade (%)', ascending=False))
else:
    print(crosst)


**Insight:** Expõe qual centro metropolitano enfrentou a pior correlação obito/mortes.
